In [ ]:
# The main reason behind using a memory system is because 
# every LLM call is a unique request, and it doesn't store any state.
# This is why you constantly need to give context to your LLM.

# for this you store the conversation and inject it with the prompt in each request
# which ultimately gives the LLM context on the conversation, make the responses more to the point and wholesome.

# To store these context messages, you can either store them in-memory or in a database.
# in-memory generally fails in production, hence database is preferred for production.

# There are various methods to use this memory system.
# 1. ConverstionBufferMemory which is now become old. 
#    Instead we use ChatMessageHistory + RunnableMessageHistory 
# This automatically stores all the conversations during a session. Hence, its more prone to context overflow,
# making the model to either hallucinate or loose the context. 
# 2. WindowMemory
#    This only store N number of messages at a time and if that limit exceeds, the older messages get trimmed.
# 3. ConversationSummaryBufferMemory
#    This allows you to summarize the previous conversation if the message history exceed a limit of tokens.
# The challenge faced here is, model takes more time to respond everytime the message history is filled up with set limited tokens.
# Inference becomes occassionally slower, as model has to summarize + return the inference.
# 4. ConversationTokenBufferMemory
#    This is similar to WindowMemory, but instead of messages, it makes the limits on tokens.
# Reason being: some messages may have more tokens, some may have lesser, where the chance of filling the context window still remains.
# Hence, you trim older tokens when the memory already has filled up with the token limit.


# IMPLEMENTATION:


In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model = "gemini-flash-latest", temperature = 0.7)
prompt = ChatPromptTemplate(
    [
        ("system", "You are a helpful AI Engineer Assistant."),
        MessagesPlaceholder(variable_name = "history"),
        ("human", "{input}")
    ]
)

in_memory = {}

from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

def get_session_history(session_id: str) -> ChatMessageHistory:
    if not session_id in in_memory:
        in_memory[session_id] = ChatMessageHistory()    
    return in_memory[session_id]

# Wrapping the chain with this ChatMessageHistory.

context_aware_chain = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_message_history",
)

config = {"configurable": {"session_id": "user_123"}}

response1 = context_aware_chain.invoke({"input": "My Name is Harsh."}, config = config)
print(response1)

response2 = context_aware_chain.invoke({"input":"What is my name?"}, config = config)
print(response2)

Hello Harsh! It's great to meet you. 

As your AI Engineer Assistant, I'm here to help you with anything related to machine learning, software development, prompt engineering, data science, or system design. 

What are we building, coding, or troubleshooting today?
Your name is Harsh! How can I help you today?


In [7]:
# WindowMemory

class WindowChatMessageHistory(ChatMessageHistory):
    def __init__(self, max_messages: int = 6):
        super().__init__()
        self.max_messages = max_messages

    def add_message(self, message):
        super().add_message(message)
        if len(self.messages) >= self.max_messages:
            self.messages = self.messages[-self.max_messages:]

window_in_memory = {}

def get_session_id(session_id: str) -> WindowChatMessageHistory:
    if not session_id in window_in_memory:
        window_in_memory[session_id] = WindowChatMessageHistory(max_messages = 6)
    return window_in_memory[session_id]

windowed_context_chain = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key= "input",
    history_messages_key= "history"
)


c:\Projects\Training\training_env\Lib\site-packages\IPython\core\interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
# ConversationSummaryBufferMemory
# already deprecated. Write custom logic to create a Summary of the History.




ModuleNotFoundError: No module named 'langchain.memory'

In [ ]:
# Trimming the chat message history
from langchain_core.messages import trim_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from operator import itemgetter

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)

# trim_messages: keeps last N messages, always keeps system message intact
trimmer = trim_messages(
    max_tokens=500,          # token budget for history
    strategy="last",         # keep most recent messages
    token_counter=llm,       # uses the model's tokenizer
    include_system=True,     # never trim the system message
    allow_partial=False,
    start_on="human"         # always start history on a human message
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful senior engineer."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Wire trimmer into the chain
chain = (
    RunnablePassthrough.assign(
        history=itemgetter("history") | trimmer
    )
    | prompt
    | llm
)

session_store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

config = {"configurable": {"session_id": "user_456"}}

r1 = chain_with_memory.invoke({"input": "I'm building a FastAPI backend."}, config=config)
print(r1.content)

r2 = chain_with_memory.invoke({"input": "What was I building?"}, config=config)
print(r2.content)

In [ ]:
# An entire production ready orchestration of context management along with storing the state specific information.

from dataclasses import dataclass, field
from typing import Optional
from langchain_community.chat_message_histories import ChatMessageHistory

# Session specific information tracking.
@dataclass
class UserSession:
    user_id: str
    history: ChatMessageHistory = field(default_factory=ChatMessageHistory)
    # metadata that helps personalize responses
    user_role: Optional[str] = None        # "junior", "senior", "architect"
    current_topic: Optional[str] = None    # "debugging", "architecture", etc.
    message_count: int = 0

# Session store storing UserSession with user_id as session_id.
session_store: dict[str, UserSession] = {}

def get_or_create_session(user_id: str) -> UserSession:
    if user_id not in session_store:
        session_store[user_id] = UserSession(user_id=user_id)
    return session_store[user_id]

def get_history_for_chain(session_id: str) -> ChatMessageHistory:
    session = get_or_create_session(session_id)
    return session.history

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Dynamic prompt building based on the user_role retrieved from the UserSession which is stored in the session_store.
def build_prompt_for_user(user_role: str = "engineer") -> ChatPromptTemplate:
    role_instructions = {
        "junior": "Use simple language. Explain concepts. Avoid jargon.",
        "senior": "Be concise. Skip basics. Focus on tradeoffs.",
        "architect": "Focus on system design, scalability, and patterns."
    }
    instruction = role_instructions.get(user_role, "Be helpful and clear.")

    return ChatPromptTemplate.from_messages([
        ("system", f"You are a senior engineering assistant. {instruction}"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])

# Usage
session = get_or_create_session("arjun_001")
session.user_role = "senior"

prompt = build_prompt_for_user(session.user_role)

chain = (
    RunnablePassthrough.assign(
        history=itemgetter("history") | trimmer
    )
    | prompt
    | llm
    | StrOutputParser()
)

chatbot = RunnableWithMessageHistory(
    chain,
    get_history_for_chain,
    input_messages_key="input",
    history_messages_key="history"
)